In [1]:
import os
import re
import sys
import glob
import time
import intake
import numpy as np
import pandas as pd
import scipy as io
import xarray as xr
import matplotlib.dates as mdates

from datetime import date, datetime, timedelta
from matplotlib import pyplot as plt
from dask.distributed import Client
from xarray.coding.times import CFDatetimeCoder

time_coder = CFDatetimeCoder(use_cftime=True)

In [2]:
client = Client()
client

/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44785 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/44785/status,
Dashboard: /proxy/44785/status,Workers: 7
Total threads: 28,Total memory: 125.19 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:35943,Workers: 0
Dashboard: /proxy/44785/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:41873,Total threads: 4
Dashboard: /proxy/33923/status,Memory: 17.88 GiB
Nanny: tcp://127.0.0.1:45925,


In [3]:
vars_SOTS = ['SW_NET', 'LW_NET', 'HL', 'HS', 'HEAT_NET']

SOTS_dir  = '/g/data/jk72/mxr581/SOTS-project-data/sots/flux'
SOTS_pattern = os.path.join(SOTS_dir, "IMOS_DWM-*_CFMST_20*.nc")

SOTS_paths = glob.glob(SOTS_pattern)

def extract_year(path):
    match = re.search(r'CFMST_(\d{4})', path)
    if match:
        return int(match.group(1))

SOTS_sorted = sorted(SOTS_paths, key=extract_year)

# ---------------------------
# OPEN ALL FILES ONCE
# ---------------------------
start = time.time()
print("Opening data…")

ds = xr.open_mfdataset(
    SOTS_sorted,
    combine='nested',
    concat_dim='TIME',
    data_vars='minimal',
    coords='minimal',
    compat='override'
)

# ---------------------------
# LOOP VARIABLES
# ---------------------------
winter_flux = {}

for var in vars_SOTS:
    print(f"Processing {var}…")
    sys.stdout.flush()

    flux = ds[var]

    # monthly climatology
    clim = flux.groupby("TIME.month").mean(dim="TIME")

    # June–August mean
    winter_flux[var] = clim.sel(month=slice(6, 8)).mean(dim="month").compute()

print("Closing dataset…")
ds.close()

end = time.time()

print("Finished all variables.")
print(f"Execution time: {(end - start)/60:.2f} minutes")
sys.stdout.flush()

Opening data…
Processing SW_NET…


/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 35.91 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Processing LW_NET…


/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 35.91 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Processing HL…


...

Processing HS…


/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 35.91 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Processing HEAT_NET…


/g/data/xp65/public/apps/med_conda/envs/analysis3-25.09/lib/python3.11/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 35.91 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Closing dataset…
Finished all variables.
Execution time: 0.91 minutes


In [4]:
winter_flux

{'SW_NET': <xarray.DataArray 'SW_NET' ()> Size: 4B
 array(-51.197308, dtype=float32)
 Coordinates:
     NOMINAL_DEPTH  float32 4B 0.0,
 'LW_NET': <xarray.DataArray 'LW_NET' ()> Size: 4B
 array(44.497814, dtype=float32)
 Coordinates:
     NOMINAL_DEPTH  float32 4B 0.0,
 'HL': <xarray.DataArray 'HL' ()> Size: 4B
 array(73.04998, dtype=float32)
 Coordinates:
     NOMINAL_DEPTH  float32 4B 0.0,
 'HS': <xarray.DataArray 'HS' ()> Size: 4B
 array(17.188444, dtype=float32)
 Coordinates:
     NOMINAL_DEPTH  float32 4B 0.0,
 'HEAT_NET': <xarray.DataArray 'HEAT_NET' ()> Size: 4B
 array(83.472244, dtype=float32)
 Coordinates:
     NOMINAL_DEPTH  float32 4B 0.0}